# **Download Dataset**

In [1]:
import pandas as pd
import urllib.request
import zipfile
import io
import ssl

# تجاهل التحقق من شهادة SSL
ssl._create_default_https_context = ssl._create_unverified_context

url = "https://archive.ics.uci.edu/static/public/228/sms+spam+collection.zip"

with urllib.request.urlopen(url) as response:
    with zipfile.ZipFile(io.BytesIO(response.read())) as z:
        with z.open("SMSSpamCollection") as f:
            df = pd.read_csv(f, sep='\t', names=['label', 'text'])

print(df.head())

  label                                               text
0   ham  Go until jurong point, crazy.. Available only ...
1   ham                      Ok lar... Joking wif u oni...
2  spam  Free entry in 2 a wkly comp to win FA Cup fina...
3   ham  U dun say so early hor... U c already then say...
4   ham  Nah I don't think he goes to usf, he lives aro...


# **Text Preprocessing**

In [2]:
import re
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

nltk.download('stopwords')
nltk.download('wordnet')

lemmatizer = WordNetLemmatizer()
stop_words = set(stopwords.words('english'))

def clean_sms_text(text):
    if not isinstance(text, str):
        return ""

    # lowercase
    text = text.lower()

    # replace urls
    text = re.sub(r'http\S+|www\S+', ' URL ', text)

    # replace money
    text = re.sub(r'[$£€]', ' MONEY ', text)

    # replace numbers
    text = re.sub(r'\d+', ' NUMBER ', text)

    # remove unwanted chars
    text = re.sub(r'[^a-z\s!]', '', text)

    # remove repeated chars
    text = re.sub(r'(.)\1{2,}', r'\1\1', text)

    # tokenize
    words = text.split()

    # remove stopwords + lemmatization
    cleaned_words = [
        lemmatizer.lemmatize(word)
        for word in words
        if word not in stop_words
    ]

    return " ".join(cleaned_words)

sample_text = """
URGENT! Your mobile No 07732584351
was awarded a £2,000 bonus prize!!
go to http://prize.com
"""

print(clean_sms_text(sample_text))

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\Moham\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\Moham\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


urgent! mobile awarded bonus prize!! go


In [3]:
# افترض إن اسم الداتا فريم df والعمود اسمه text
# الكود ده هيعمل عمود جديد اسمه cleaned_text فيه النص النظيف جاهز
df['cleaned_text'] = df['text'].apply(clean_sms_text)

# ألقِ نظرة على أول 5 صفوف لتتأكد
print(df[['text', 'cleaned_text']].head())

                                                text  \
0  Go until jurong point, crazy.. Available only ...   
1                      Ok lar... Joking wif u oni...   
2  Free entry in 2 a wkly comp to win FA Cup fina...   
3  U dun say so early hor... U c already then say...   
4  Nah I don't think he goes to usf, he lives aro...   

                                        cleaned_text  
0  go jurong point crazy available bugis n great ...  
1                            ok lar joking wif u oni  
2  free entry wkly comp win fa cup final tkts st ...  
3                u dun say early hor u c already say  
4           nah dont think go usf life around though  


# **Train-Test Split**

In [4]:
from sklearn.model_selection import train_test_split

# 1. تحويل الـ Labels لأرقام (ham -> 0, spam -> 1)
# تأكد إن اسم العمود في الداتا سيت عندك 'label'، لو كان 'v1' غيّره في الكود
df['label_numeric'] = df['label'].map({'ham': 0, 'spam': 1})

# تحديد المدخلات (النص التنظيف) والمخرجات (الرقم)
X = df['cleaned_text']
y = df['label_numeric']

# 2. تقسيم الداتا لتدريب واختبار
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# طباعة الحجم للتأكد من التقسيم
print(f"--- تم تقسيم البيانات بنجاح ---")
print(f"عدد رسائل التدريب (X_train): {len(X_train)}")
print(f"عدد رسائل الاختبار (X_test): {len(X_test)}")

--- تم تقسيم البيانات بنجاح ---
عدد رسائل التدريب (X_train): 4457
عدد رسائل الاختبار (X_test): 1115


# **TF-IDF Vectorization**

In [5]:
from sklearn.feature_extraction.text import TfidfVectorizer

# 1. تعريف الـ Vectorizer وتحديد الحد الأقصى للكلمات بـ 3000 كلمة
tfidf = TfidfVectorizer(max_features=3000)

# 2. بناء قاموس الكلمات وتحويل بيانات التدريب (Fit + Transform)
X_train_tfidf = tfidf.fit_transform(X_train)

# 3. تحويل بيانات الاختبار بناءً على نفس القاموس (Transform فقط)
X_test_tfidf = tfidf.transform(X_test)

# طباعة أبعاد المصفوفات الناتجة (عدد الرسائل × 3000 ميزة)
print(f"--- تم تحويل النصوص لأرقام بنجاح ---")
print(f"حجم مصفوفة التدريب الرقمية: {X_train_tfidf.shape}")
print(f"حجم مصفوفة الاختبار الرقمية: {X_test_tfidf.shape}")

--- تم تحويل النصوص لأرقام بنجاح ---
حجم مصفوفة التدريب الرقمية: (4457, 3000)
حجم مصفوفة الاختبار الرقمية: (1115, 3000)


# **Machine Learning Models** 

 **Decision Tree**

In [6]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import classification_report

dt_model = DecisionTreeClassifier(
    max_depth=20,
    min_samples_split=5,
    min_samples_leaf=2,
    max_features='sqrt',
    random_state=42
)

dt_model.fit(X_train_tfidf, y_train)

dt_preds = dt_model.predict(X_test_tfidf)

print(classification_report(
    y_test,
    dt_preds,
    target_names=['Ham', 'Spam']
))

              precision    recall  f1-score   support

         Ham       0.93      0.99      0.96       966
        Spam       0.86      0.49      0.62       149

    accuracy                           0.92      1115
   macro avg       0.89      0.74      0.79      1115
weighted avg       0.92      0.92      0.91      1115



In [7]:
train_acc = dt_model.score(X_train_tfidf, y_train)
test_acc = dt_model.score(X_test_tfidf, y_test)

print(train_acc)
print(test_acc)

0.9295490240071798
0.9210762331838565


**XGBoost**

In [8]:
from xgboost import XGBClassifier
from sklearn.metrics import (
    classification_report,
    accuracy_score
)

print("\nجاري تدريب موديل XGBoost...")

xgb_model = XGBClassifier(
    random_state=42,
    eval_metric='logloss',
    n_estimators=100,
    max_depth=4,
    learning_rate=0.1,
    subsample=0.8,
    colsample_bytree=0.8
)

xgb_model.fit(X_train_tfidf, y_train)

xgb_preds = xgb_model.predict(X_test_tfidf)

print("\n📊 --- تقرير أداء XGBoost ---")
print(classification_report(
    y_test,
    xgb_preds,
    target_names=['Ham', 'Spam']
))

# Overfitting Check
train_acc = accuracy_score(
    y_train,
    xgb_model.predict(X_train_tfidf)
)

test_acc = accuracy_score(
    y_test,
    xgb_preds
)

print(f"\nTraining Accuracy: {train_acc:.4f}")
print(f"Testing Accuracy : {test_acc:.4f}")


جاري تدريب موديل XGBoost...

📊 --- تقرير أداء XGBoost ---
              precision    recall  f1-score   support

         Ham       0.96      0.99      0.98       966
        Spam       0.95      0.74      0.83       149

    accuracy                           0.96      1115
   macro avg       0.95      0.87      0.90      1115
weighted avg       0.96      0.96      0.96      1115


Training Accuracy: 0.9753
Testing Accuracy : 0.9596


# **Deep Learning**

**CNN + LSTM**

In [9]:
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

tokenizer = Tokenizer(num_words=5000, oov_token="<OOV>")
tokenizer.fit_on_texts(X_train)
X_train_seq = tokenizer.texts_to_sequences(X_train)
X_test_seq = tokenizer.texts_to_sequences(X_test)
X_train_pad = pad_sequences(X_train_seq, maxlen=50, padding='post', truncating='post')
X_test_pad = pad_sequences(X_test_seq, maxlen=50, padding='post', truncating='post')

In [10]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, Conv1D, MaxPooling1D, LSTM, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from tensorflow.keras.regularizers import l2

In [11]:
model = Sequential([
    
    Embedding(input_dim=5000, output_dim=128, input_length=50),

    Conv1D(
        filters=64,
        kernel_size=3,
        activation='relu',
        kernel_regularizer=l2(0.001)
    ),
    MaxPooling1D(pool_size=2),
    Dropout(0.3),

    LSTM(
        units=64,
        kernel_regularizer=l2(0.001)
    ),
    Dropout(0.5),

    Dense(1, activation='sigmoid')
])

C:\Users\Moham\AppData\Roaming\Python\Python310\site-packages\keras\src\layers\core\embedding.py:97: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


In [12]:
model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

In [13]:
early_stop = EarlyStopping(
    monitor='val_loss',
    patience=1,
    restore_best_weights=True
)

lr_reduce = ReduceLROnPlateau(
    monitor='val_loss',
    factor=0.5,
    patience=1
)

In [14]:
history = model.fit(
    X_train_pad,
    y_train,
    epochs=10,
    batch_size=32,
    validation_split=0.1,
    callbacks=[early_stop, lr_reduce]
)

Epoch 1/10
126/126 ━━━━━━━━━━━━━━━━━━━━ 6s 25ms/step - accuracy: 0.8935 - loss: 0.4128 - val_accuracy: 0.9507 - val_loss: 0.2855 - learning_rate: 0.0010
Epoch 2/10
126/126 ━━━━━━━━━━━━━━━━━━━━ 2s 18ms/step - accuracy: 0.9820 - loss: 0.1197 - val_accuracy: 0.9798 - val_loss: 0.1158 - learning_rate: 0.0010
Epoch 3/10
126/126 ━━━━━━━━━━━━━━━━━━━━ 2s 17ms/step - accuracy: 0.9923 - loss: 0.0619 - val_accuracy: 0.9821 - val_loss: 0.0929 - learning_rate: 0.0010
Epoch 4/10
126/126 ━━━━━━━━━━━━━━━━━━━━ 2s 17ms/step - accuracy: 0.9940 - loss: 0.0439 - val_accuracy: 0.9865 - val_loss: 0.0963 - learning_rate: 0.0010
